<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/notebook/05_entrenamiento_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Entrenamiento en inglés (mBERT, XLM-RoBERTa)

Este notebook entrena tres modelos sobre el conjunto de entrenamiento
traducido al español: mBERT, XLM-RoBERTa y BETO. BETO se incluye
únicamente en esta fase, al ser un modelo monolingüe en español, para
contrastarlo frente a los modelos multilingües entrenados con el mismo
conjunto de datos.

Por el momento se usa la Parte 1 del conjunto traducido (NLLB-200),
mientras se completa la traducción de las partes restantes.

In [ ]:
!pip install transformers datasets scikit-learn accelerate -q

In [1]:
import pandas as pd
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Dispositivo:', device)

Dispositivo: cuda


In [ ]:
## 2. Conexión a Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Carga del conjunto de entrenamiento traducido

Ajustar `RUTA_DATASET` al nombre exacto del archivo final una vez que
las 3 partes traducidas estén unidas y subidas a Drive.

In [3]:
RUTA_DATASET = '/content/drive/MyDrive/dataset_limpio_ingles_balanceado.csv'

df = pd.read_csv(RUTA_DATASET)
print('Filas cargadas:', len(df))
print('Columnas:', df.columns.tolist())
print()
print('Distribución de etiquetas:')
print(df['label'].value_counts())

Filas cargadas: 9324
Columnas: ['text', 'label']

Distribución de etiquetas:
label
1    4662
0    4662
Name: count, dtype: int64


## 4. División en entrenamiento, validación y prueba

División estratificada (80/10/10), de forma que la proporción de
clases se mantenga igual en las tres particiones.

In [4]:
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp['label'], random_state=42
)

print('Entrenamiento:', len(df_train))
print('Validación:', len(df_val))
print('Prueba:', len(df_test))

Entrenamiento: 7459
Validación: 932
Prueba: 933


## 5. Cálculo de pesos de clase

El conjunto ya está balanceado desde la etapa de limpieza, pero se
calculan los pesos de clase como práctica robusta, para que el
entrenamiento siga siendo correcto incluso si en el futuro se usa este
mismo notebook con un dataset que no esté perfectamente balanceado.

In [5]:
from sklearn.utils.class_weight import compute_class_weight

pesos_clase = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train['label']),
    y=df_train['label']
)
pesos_clase_tensor = torch.tensor(pesos_clase, dtype=torch.float).to(device)

print('Pesos de clase calculados:', pesos_clase)

Pesos de clase calculados: [1.00013408 0.99986595]


## 6. Función de entrenamiento

Se define una función reutilizable que recibe el nombre del modelo
preentrenado y devuelve el modelo ajustado junto con sus métricas sobre
el conjunto de prueba interno. Usa los pesos de clase calculados en el
paso anterior dentro de una función de pérdida personalizada, en vez de
la pérdida por defecto del `Trainer`.

In [6]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn as nn


def calcular_metricas(eval_pred):
    logits, labels = eval_pred
    predicciones = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicciones, average='binary'
    )
    accuracy = accuracy_score(labels, predicciones)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


class EntrenadorConPesos(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        funcion_perdida = nn.CrossEntropyLoss(weight=pesos_clase_tensor)
        loss = funcion_perdida(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 7. Entrenamiento de mBERT

In [7]:
nombre_modelo = 'bert-base-multilingual-cased'
nombre_corto = 'mbert_es'

tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

def tokenizar(ejemplos):
    return tokenizer(ejemplos['text'], truncation=True, padding='max_length', max_length=512)

ds_train = Dataset.from_pandas(df_train[['text', 'label']]).map(tokenizar, batched=True)
ds_val = Dataset.from_pandas(df_val[['text', 'label']]).map(tokenizar, batched=True)
ds_test = Dataset.from_pandas(df_test[['text', 'label']]).map(tokenizar, batched=True)

modelo_mbert = AutoModelForSequenceClassification.from_pretrained(nombre_modelo, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

Map:   0%|          | 0/932 [00:00<?, ? examples/s]

Map:   0%|          | 0/933 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
args_entrenamiento = TrainingArguments(
    output_dir=f'./resultados_{nombre_corto}',
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir=f'./logs_{nombre_corto}',
    report_to='none',
)

entrenador_mbert = EntrenadorConPesos(
    model=modelo_mbert,
    args=args_entrenamiento,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=calcular_metricas,
)

entrenador_mbert.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.155445,0.019920,0.996781,0.997849,0.995708,0.996778
2,0.031844,0.031809,0.994635,0.993576,0.995708,0.994641
3,0.009085,0.026566,0.995708,0.995708,0.995708,0.995708


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2799, training_loss=0.04955567422190833, metrics={'train_runtime': 2546.0357, 'train_samples_per_second': 8.789, 'train_steps_per_second': 1.099, 'total_flos': 5887636085790720.0, 'train_loss': 0.04955567422190833, 'epoch': 3.0})

## 8. Evaluación de mBERT sobre el conjunto de prueba interno

In [10]:
metricas_mbert = entrenador_mbert.evaluate(ds_test)
print('--- Métricas de mBERT en prueba interna ---')
print(metricas_mbert)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.009085,0.029552,3,0.993569,0.995690,0.991416,0.993548


--- Métricas de mBERT en prueba interna ---
{'eval_loss': 0.029551533982157707, 'eval_accuracy': 0.9935691318327974, 'eval_precision': 0.9956896551724138, 'eval_recall': 0.9914163090128756, 'eval_f1': 0.9935483870967742}


## 9. Guardado del modelo mBERT entrenado

In [11]:
entrenador_mbert.save_model('/content/drive/MyDrive/modelo_mbert_en')
print('Modelo mBERT guardado en Drive.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo mBERT guardado en Drive.
